# 🧾 Bill Management Agent

[![Python](https://img.shields.io/badge/Python-3.12%2B-blue?logo=python&logoColor=white)](https://www.python.org/)
[![AutoGen](https://img.shields.io/badge/AutoGen-AgentChat%200.7-purple?logo=microsoft&logoColor=white)](https://github.com/microsoft/autogen)
[![Gemini](https://img.shields.io/badge/Google-Gemini%202.0%20Flash-4285F4?logo=google&logoColor=white)](https://ai.google.dev/)
[![License](https://img.shields.io/badge/License-MIT-green)](LICENSE)

**Objective:** Build a multi-agent bill management system using AutoGen's `RoundRobinGroupChat` that processes receipt images, categorizes expenses, and delivers actionable spending insights — powered by Google Gemini 2.0 Flash.

> Workflow diagram: [`Flow/workflow.mmd`](Flow/workflow.mmd)

---

## 🤖 Agent Descriptions

| Agent | Role | AutoGen Class |
|-------|------|---------------|
| **Bill Processing Agent** | Extracts all line items from the bill image and organizes them into expense categories | `AssistantAgent` |
| **Expense Summarization Agent** | Analyses categorized expenses, computes totals, flags high-spend items, and provides insights | `AssistantAgent` |

---

## 🔄 Pipeline Flow

1. A bill image is encoded as base64 and embedded in the task message
2. `RoundRobinGroupChat` routes the task to **Bill_Processing_Agent** first
3. The agent extracts line items, assigns categories, and outputs a structured JSON block
4. The chat moves to **Expense_Summarization_Agent**, which reads the extracted data
5. The summarizer produces a full spending report and ends with `BILL_ANALYSIS_COMPLETE`
6. `TextMentionTermination` detects the keyword and stops the pipeline

---

## Table of Contents
1. [Imports & Environment Setup](#1-imports--environment-setup)
2. [LLM Configuration — Gemini 2.0 Flash](#2-llm-configuration)
3. [Generate Sample Receipt Image](#3-generate-sample-receipt-image)
4. [Encode Image as Base64](#4-encode-image-as-base64)
5. [Define Agents](#5-define-agents)
6. [Build Group Chat Team](#6-build-group-chat-team)
7. [Run the Pipeline](#7-run-the-pipeline)
8. [Agent Flow Summary](#8-agent-flow-summary)

---
## 1. Imports & Environment Setup

All dependencies are listed in [`requirements.txt`](requirements.txt). Install with:
```bash
pip install -r requirements.txt
```

Set your Gemini API key in `.env`:
```
GOOGLE_GEMINI_API_KEY=your_key_here
```

In [ ]:
import asyncio
import base64
import os
from datetime import datetime

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core.models import ModelFamily
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
from IPython.display import display, Image as IPImage, Markdown
from PIL import Image, ImageDraw, ImageFont

# Output directories
DATA_DIR = "Data"
os.makedirs(DATA_DIR, exist_ok=True)

load_dotenv()

GEMINI_API_KEY = os.getenv("GOOGLE_GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GOOGLE_GEMINI_API_KEY not found. Add it to your .env file.")

print("✅ Libraries imported successfully!")
print(f"📅 Session started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔑 GOOGLE_GEMINI_API_KEY loaded: {'*' * (len(GEMINI_API_KEY) - 4) + GEMINI_API_KEY[-4:]}")

---
## 2. LLM Configuration

Gemini 2.0 Flash is accessed via its **OpenAI-compatible endpoint**, which lets us use `OpenAIChatCompletionClient` from `autogen-ext` without any extra SDK.
The `model_info` dict tells AutoGen about the model's capabilities (vision, no function calling, etc.).

In [ ]:
# ─── Gemini via OpenAI-compatible endpoint ────────────────────────────────────
def get_gemini_client() -> OpenAIChatCompletionClient:
    return OpenAIChatCompletionClient(
        model="models/gemini-2.0-flash",
        api_key=os.getenv("GOOGLE_GEMINI_API_KEY"),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        model_info={
            "family": ModelFamily.GEMINI_2_0_FLASH,
            "function_calling": False,
            "json_output": False,
            "vision": True,
            "structured_output": False,
        },
    )

print("✅ Gemini 2.0 Flash client configured!")
print("   Model    : models/gemini-2.0-flash")
print("   Endpoint : generativelanguage.googleapis.com/v1beta/openai/")
print("   Vision   : enabled")

---
## 3. Generate Sample Receipt Image

We programmatically generate a realistic supermarket receipt using `Pillow`.
This simulates a real-world bill image that the agent pipeline will process.

> To use your own bill, replace `IMAGE_PATH` with the path to your image file.

In [ ]:
# ================================================================
# GENERATE A REALISTIC SUPERMARKET RECEIPT
# ================================================================
DATA_DIR = DATA_DIR if 'DATA_DIR' in dir() else 'Data'
os.makedirs(DATA_DIR, exist_ok=True)

def create_sample_receipt(path: str = 'sample_bill.png') -> str:
    """
    Draws a supermarket receipt image using Pillow.

    Args:
        path: Output file path for the generated receipt image.

    Returns:
        Path to the saved image.
    """
    W, H = 400, 720
    img  = Image.new('RGB', (W, H), color='#FAFAF8')
    draw = ImageDraw.Draw(img)

    try:
        font_b = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 16)
        font_r = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 13)
        font_s = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 11)
    except:
        font_b = font_r = font_s = ImageFont.load_default()

    # ── Header ────────────────────────────────────────────────────────────────
    draw.rectangle([0, 0, W, 80], fill='#1a1a2e')
    draw.text((W//2, 25), 'FRESH MART SUPERSTORE',           font=font_b, fill='#FFD700', anchor='mm')
    draw.text((W//2, 50), '42 Market Street, Mumbai',        font=font_s, fill='#CCC',    anchor='mm')
    draw.text((W//2, 65), 'Date: 15 Mar 2025  Bill: INV-0342', font=font_s, fill='#AAA',  anchor='mm')

    # ── Column headers ────────────────────────────────────────────────────────
    y = 95
    draw.line([(15, y), (W-15, y)], fill='#CCC')
    y += 8
    draw.text((20,  y), 'ITEM', font=font_b, fill='#222')
    draw.text((250, y), 'QTY',  font=font_b, fill='#222')
    draw.text((300, y), 'RATE', font=font_b, fill='#222')
    draw.text((355, y), 'AMT',  font=font_b, fill='#222')
    y += 20

    # ── Line items ────────────────────────────────────────────────────────────
    items = [
        ('Organic Basmati Rice 5kg',  2, 285, 570),
        ('Amul Butter 500g',          3,  58, 174),
        ('Tata Salt 1kg',             2,  22,  44),
        ('Dove Shampoo 200ml',        1, 185, 185),
        ('Tropicana Orange Juice 1L', 2, 120, 240),
        ('Britannia Bread',           2,  42,  84),
        ('Lays Chips Family Pack',    1,  95,  95),
        ('Colgate Toothpaste 200g',   1,  78,  78),
        ('Nestle KitKat 4-Pack',      2,  55, 110),
        ('Fresh Paneer 500g',         1, 145, 145),
        ('Parle-G Biscuits 800g',     1,  68,  68),
        ('Detergent Powder 2kg',      1, 220, 220),
    ]
    for name, qty, rate, amt in items:
        short = name[:28] + '..' if len(name) > 28 else name
        draw.text((20,  y), short,     font=font_s, fill='#1a1a2e')
        draw.text((252, y), str(qty),  font=font_s, fill='#333')
        draw.text((292, y), str(rate), font=font_s, fill='#333')
        draw.text((348, y), str(amt),  font=font_s, fill='#111')
        y += 18

    # ── Totals ────────────────────────────────────────────────────────────────
    subtotal = sum(i[3] for i in items)
    tax      = round(subtotal * 0.05, 2)
    total    = subtotal + tax

    y += 8
    draw.line([(15, y), (W-15, y)], fill='#AAA')
    y += 10
    draw.text((20,  y), 'Subtotal:', font=font_r, fill='#333')
    draw.text((370, y), f'Rs.{subtotal:.2f}', font=font_r, fill='#333', anchor='ra')
    y += 20
    draw.text((20,  y), 'GST (5%):', font=font_r, fill='#333')
    draw.text((370, y), f'Rs.{tax:.2f}', font=font_r, fill='#333', anchor='ra')
    y += 20
    draw.rectangle([15, y, W-15, y+30], fill='#1a1a2e')
    draw.text((25,  y+8), 'TOTAL:', font=font_b, fill='#FFD700')
    draw.text((375, y+8), f'Rs.{total:.2f}', font=font_b, fill='#FFD700', anchor='ra')
    y += 50
    draw.text((W//2, y), 'Thank you for shopping!', font=font_s, fill='#888', anchor='mm')

    img.save(path)
    print(f'✅ Sample receipt created: {path}')
    print(f'   Items: {len(items)} | Subtotal: Rs.{subtotal} | GST: Rs.{tax} | Total: Rs.{total:.2f}')
    return path


IMAGE_PATH = create_sample_receipt(f"{DATA_DIR}/sample_bill.png")
display(IPImage(IMAGE_PATH, width=320))

---
## 4. Encode Image as Base64

Gemini's vision API accepts images as base64-encoded data URIs embedded directly in the message content.
We encode the receipt image here so it can be passed as part of the multimodal task message.

In [ ]:
# ─── Encode the receipt image for the multimodal message ─────────────────────
def image_to_base64(path: str) -> tuple:
    """
    Encodes an image file as a base64 string.

    Args:
        path: Path to the image file.

    Returns:
        Tuple of (base64_string, media_type).
    """
    import mimetypes
    mime = mimetypes.guess_type(path)[0] or 'image/jpeg'
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8'), mime


img_b64, img_mime = image_to_base64(IMAGE_PATH)
print(f'✅ Image encoded successfully!')
print(f'   Size      : {len(img_b64):,} characters')
print(f'   MIME type : {img_mime}')

---
## 5. Define Agents

Each agent is an `AssistantAgent` backed by the Gemini 2.0 Flash client.
Their `system_message` defines their role, expected input, output format, and termination signal.

In [ ]:
# ================================================================
# AGENT 1: BILL PROCESSING AGENT
# Role: Extract line items from the bill image and categorize them
# ================================================================
bill_processing_agent = AssistantAgent(
    name="Bill_Processing_Agent",
    model_client=get_gemini_client(),
    system_message="""You are the Bill Processing Agent.
You receive bill/receipt images and extract ALL expense data.

Categories: Groceries | Dining | Utilities | Shopping | Entertainment | Healthcare | Transport | Other

Tasks:
1. Extract vendor name, date, and every line item from the bill.
2. Assign each item to the most appropriate category.
3. Calculate subtotal, tax, and grand total.

Output format:
## Extracted Bill Data
**Vendor:** X  |  **Date:** X  |  **Bill Type:** X

| Item | Category | Qty | Rate | Amount |
|------|----------|-----|------|--------|

**Subtotal:** X  |  **Tax:** X  |  **Total:** X

Then a ```json block with keys: vendor, date, bill_category, items[], subtotal, tax, total.

End with: Passing to Expense_Summarization_Agent.""",
)

print("✅ Bill_Processing_Agent defined!")


# ================================================================
# AGENT 2: EXPENSE SUMMARIZATION AGENT
# Role: Analyse categorized expenses and generate a spending report
# ================================================================
expense_summarization_agent = AssistantAgent(
    name="Expense_Summarization_Agent",
    model_client=get_gemini_client(),
    system_message="""You are the Expense Summarization Agent.
You analyse the categorized expense data from the Bill Processing Agent.

Tasks:
1. Calculate total and per-category breakdown with % share.
2. Identify the highest-spending category.
3. Flag unusual or high-value items (> 20% of total).
4. Provide 3-4 actionable spending insights.
5. Give an overall budget health assessment.

Output format:
# Expense Summary Report
## Overview
## Spending Breakdown by Category
| Category | Amount | % Share | Status |
|----------|--------|---------|--------|
## Highest Expenditure Category
## Unusual or High-Value Items
## Spending Insights & Recommendations
## Budget Health Assessment

End with: BILL_ANALYSIS_COMPLETE""",
)

print("✅ Expense_Summarization_Agent defined!")
print()
print("   Bill_Processing_Agent      → extracts & categorizes line items")
print("   Expense_Summarization_Agent → analyses spend & generates report")

---
## 6. Build Group Chat Team

`RoundRobinGroupChat` runs agents in a fixed sequential order:
`Bill_Processing_Agent` → `Expense_Summarization_Agent`.

The pipeline stops automatically when `BILL_ANALYSIS_COMPLETE` appears in any message,
or after a maximum of 10 messages as a safety fallback.

In [ ]:
# ─── Termination conditions ───────────────────────────────────────────────────
# Stop when Expense_Summarization_Agent signals completion, or after 10 messages
termination = TextMentionTermination("BILL_ANALYSIS_COMPLETE") | MaxMessageTermination(10)

# ─── RoundRobinGroupChat ──────────────────────────────────────────────────────
# Agents take turns in the order they are listed
team = RoundRobinGroupChat(
    participants=[bill_processing_agent, expense_summarization_agent],
    termination_condition=termination,
)

print("✅ RoundRobinGroupChat team ready!")
print(f"   Participants : {[a.name for a in team._participants]}")
print(f"   Order        : Bill_Processing_Agent → Expense_Summarization_Agent")
print(f"   Termination  : TextMentionTermination('BILL_ANALYSIS_COMPLETE') | MaxMessageTermination(10)")

---
## 7. Run the Pipeline

The task message is a **multimodal list** containing:
- A text instruction describing the two-step pipeline
- The bill image wrapped in `autogen_core.Image` (loaded from base64)

`Console` streams each agent's response to the output as it arrives.

In [ ]:
# ─── Build multimodal task message ───────────────────────────────────────────
import asyncio
from autogen_agentchat.messages import MultiModalMessage
from autogen_agentchat.ui import Console
from autogen_core import Image as AGImage
from pathlib import Path

# Load image directly — no dependency on img_b64 from a previous cell
_image_path = IMAGE_PATH if 'IMAGE_PATH' in dir() else 'Data/sample_bill.png'
ag_image = AGImage.from_file(Path(_image_path))

task = MultiModalMessage(
    source="user",
    content=[
        "Please process this supermarket bill image.\n"
        "Step 1 — Bill_Processing_Agent: extract all items, assign categories, compute totals.\n"
        "Step 2 — Expense_Summarization_Agent: analyse categories and generate a spending report.",
        ag_image,
    ],
)

# ─── Retry wrapper for 429 / RateLimitError ──────────────────────────────────
async def run_with_retry(team, task, max_retries: int = 5, base_delay: int = 30):
    """Run the pipeline, retrying on 429 with exponential backoff."""
    for attempt in range(1, max_retries + 1):
        try:
            await Console(team.run_stream(task=task))
            return  # success
        except RuntimeError as e:
            if '429' in str(e) or 'RateLimitError' in str(e):
                wait = base_delay * attempt
                print(f"⚠️  Rate limit hit (attempt {attempt}/{max_retries}). Retrying in {wait}s...")
                await asyncio.sleep(wait)
                # Re-build team for fresh agent state
                await team.reset()
            else:
                raise
    print("❌ Max retries reached. Please wait for your quota to reset and try again.")

print("🚀 Starting Bill Management Agent pipeline...")
print("=" * 60)

await run_with_retry(team, task)

print("=" * 60)
print("✅ Bill analysis complete!")

---
## 8. Agent Flow Summary

In [ ]:
summary = """
## 🗨️ Bill Management Agent — AutoGen Group Chat Flow

| Step | Agent | Role | AutoGen Class |
|------|-------|------|---------------|
| 1 | User (task message) | Provides bill image + pipeline instructions | `RoundRobinGroupChat.run_stream()` |
| 2 | Bill_Processing_Agent | Extracts items via Gemini Vision, categorizes expenses | `AssistantAgent` |
| 3 | Expense_Summarization_Agent | Analyses categories, generates spending report | `AssistantAgent` |

### AutoGen Components Used
- `autogen_agentchat.agents.AssistantAgent` — both processing and summarization agents
- `autogen_agentchat.teams.RoundRobinGroupChat` — sequential agent orchestration
- `autogen_agentchat.conditions.TextMentionTermination` — stops on `BILL_ANALYSIS_COMPLETE`
- `autogen_agentchat.conditions.MaxMessageTermination` — safety fallback after 10 messages
- `autogen_ext.models.openai.OpenAIChatCompletionClient` — Gemini via OpenAI-compatible endpoint

### LLM
- Model    : `models/gemini-2.0-flash`
- Endpoint : `https://generativelanguage.googleapis.com/v1beta/openai/`
- Vision   : enabled (base64 image_url in task message)

### Repository Structure
```
agentic-expense-tracker/
├── bill_management_agent.ipynb   # This notebook
├── Flow/
│   └── workflow.mmd              # Mermaid workflow diagram
├── requirements.txt
├── .env.example
├── .gitignore
└── LICENSE
```
"""
display(Markdown(summary))